### her2+dish疊合圖處理:測試演算法

In [1]:
import cv2
import numpy as np
from skimage.color import rgb2hed
from skimage.exposure import rescale_intensity
from stardist.models import StarDist2D
from csbdeep.utils import normalize

In [2]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if len(tf.config.list_physical_devices('GPU')) > 0:
    print("GPU is available")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is NOT available. Running on CPU.")

Num GPUs Available:  1
GPU is available
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
def process_her2_dish_image(img_path):
    # 1. 讀取影像
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ---------------------------------------------------------
    # 步驟一：色彩分離 (Color Deconvolution)
    # 將 RGB 轉換為 HED 空間 (Hematoxylin, Eosin/DAB, DAB)
    # scikit-image 的 rgb2hed 針對 H&E 優化，但對 H&DAB (棕色) 效果通常也不錯
    # Channel 0: Hematoxylin (細胞核 - 藍紫色)
    # Channel 2: DAB (HER2 蛋白 - 棕色)
    # ---------------------------------------------------------
    hed = rgb2hed(img_rgb)

    # 提取細胞核通道 (Hematoxylin)
    nuclei_channel = hed[:, :, 0]
    # 提取 HER2 膜通道 (DAB)
    her2_membrane_channel = hed[:, :, 2]

    # 正規化這些通道以便後續處理 (轉回 0-255)
    nuclei_img_vis = (rescale_intensity(nuclei_channel, out_range=(0, 255))).astype(np.uint8)
    her2_membrane_vis = (rescale_intensity(her2_membrane_channel, out_range=(0, 255))).astype(np.uint8)

    # ---------------------------------------------------------
    # 步驟二：細胞核分割 (使用分離出來的藍色通道)
    # ---------------------------------------------------------
    # 載入 StarDist 模型
    model = StarDist2D.from_pretrained('2D_versatile_he')

    # 因為 StarDist 預設吃 RGB，我們把單色通道疊成 3 層偽裝成 RGB
    # 這裡我們用 trick：把分離出的細胞核圖當作輸入
    # 為了增強效果，可以做一點對比度增強
    img_input = np.stack([nuclei_img_vis]*3, axis=-1)
    labels, _ = model.predict_instances(normalize(img_input))

    # ---------------------------------------------------------
    # 步驟三：定義 HER2 陽性區域 (使用分離出來的棕色通道)
    # ---------------------------------------------------------
    # 設定閾值，抓出棕色夠深的地方 (你需要自己 print 圖出來調這個 100)
    _, her2_positive_mask = cv2.threshold(her2_membrane_vis, 100, 255, cv2.THRESH_BINARY)

    # ---------------------------------------------------------
    # 步驟四：訊號偵測 (紅點與黑點) - 回到原圖處理
    # ---------------------------------------------------------
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # 黑點 (HER2 Gene) - 限制飽和度避免棕色背景
    mask_black = cv2.inRange(hsv, (0, 0, 0), (180, 150, 110))

    # 紅點 (CEP17 Gene) - 放寬範圍以捕捉粉紅色調
    mask_red1 = cv2.inRange(hsv, (0, 40, 40), (15, 255, 255))
    mask_red2 = cv2.inRange(hsv, (160, 40, 40), (180, 255, 255))
    mask_red = cv2.bitwise_or(mask_red1, mask_red2)

    # Debug 輸出
    cv2.imwrite('process/output/debug_mask_black.jpg', mask_black)
    cv2.imwrite('process/output/debug_mask_red.jpg', mask_red)

    # ---------------------------------------------------------
    # 步驟五：整合計算
    # ---------------------------------------------------------
    results = []

    for nuclei_id in np.unique(labels):
        if nuclei_id == 0: continue # 背景

        # 取得單顆細胞核遮罩
        nucleus_mask = (labels == nuclei_id).astype(np.uint8) * 255

        # 擴張細胞核遮罩，包住邊緣的紅黑點
        kernel = np.ones((5,5), np.uint8)
        dilated_nucleus_mask = cv2.dilate(nucleus_mask, kernel, iterations=1)

        # 檢查這顆細胞是否在 HER2 陽性區域
        overlap = cv2.bitwise_and(her2_positive_mask, her2_positive_mask, mask=dilated_nucleus_mask)
        if cv2.countNonZero(overlap) > 10:

            # 使用擴張後的遮罩來算紅黑點
            roi_black = cv2.bitwise_and(mask_black, mask_black, mask=dilated_nucleus_mask)
            roi_red = cv2.bitwise_and(mask_red, mask_red, mask=dilated_nucleus_mask)

            # 計算連通區域 (算顆數)
            n_her2 = max(0, cv2.connectedComponents(roi_black)[0] - 1)
            n_cep17 = max(0, cv2.connectedComponents(roi_red)[0] - 1)

            results.append({
                "id": nuclei_id,
                "her2_dots": n_her2,
                "cep17_dots": n_cep17,
                "ratio": n_her2 / n_cep17 if n_cep17 > 0 else 0
            })

    return results, nuclei_img_vis, her2_membrane_vis

In [4]:
# 測試用：記得把產出的 nuclei_img_vis 和 her2_membrane_vis 存成圖片看看有沒有分乾淨
results, n_img, h_img = process_her2_dish_image('process/tile_x153600_y59392.tiff')
print(results)
cv2.imwrite('process/output/debug_nuclei.jpg', n_img)
cv2.imwrite('process/output/debug_her2.jpg', h_img)

Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
[{'id': 11, 'her2_dots': 1, 'cep17_dots': 1, 'ratio': 1.0}, {'id': 14, 'her2_dots': 6, 'cep17_dots': 1, 'ratio': 6.0}, {'id': 66, 'her2_dots': 0, 'cep17_dots': 1, 'ratio': 0.0}, {'id': 138, 'her2_dots': 0, 'cep17_dots': 20, 'ratio': 0.0}]


True

In [5]:
def debug_signal_masks(img_path):
    img = cv2.imread(img_path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # 原始設定 (可能太嚴格)
    # lower_black = np.array([0, 0, 0])
    # upper_black = np.array([180, 255, 60])

    # --- 調整建議 1：放寬黑色的亮度 (Value) ---
    # 很多黑點其實是深灰色，亮度(V)可能在 60-100 之間
    # 飽和度(S)如果太高，可能是棕色，所以限制 S 不要太高 (例如 < 200) 比較能避開深棕色
    lower_black_debug = np.array([0, 0, 0])
    upper_black_debug = np.array([180, 255, 80]) # 試著把 60 改成 80 或 90

    mask_black = cv2.inRange(hsv, lower_black_debug, upper_black_debug)

    # 存出來看！
    cv2.imwrite("process/output/debug_mask_black.jpg", mask_black)
    print("已儲存 debug_mask_black.jpg，請檢查這張圖是否全黑？")
debug_signal_masks("process/tile_x153600_y59392.tiff")

已儲存 debug_mask_black.jpg，請檢查這張圖是否全黑？


In [10]:
# --- 新增：自動計算黑色閾值的函式 ---
def get_dynamic_black_threshold(hsv_img, percentile=2):
    """
    自動分析圖片的亮度分佈，找出「最黑」的那些點。
    percentile: 設定要抓取最黑的前多少百分比的像素 (預設 0.5%)
    """
    # 提取亮度通道 (Value)
    v_channel = hsv_img[:, :, 2]

    # 提取飽和度通道 (Saturation)
    s_channel = hsv_img[:, :, 1]

    # 策略 1: 找出亮度 (V) 的動態閾值
    # 我們只看有組織的地方(忽略純白背景)，以免影響統計
    valid_pixels = v_channel[v_channel < 240]
    if len(valid_pixels) == 0:
        return 0, 100 # 防呆

    # 計算分位數：找出最暗的 X% 的數值是多少
    # 例如：如果有黑點，它們通常是這張圖裡最暗的 1%
    dynamic_v_max = np.percentile(valid_pixels, percentile)

    # 為了避免抓到太多雜訊，給它一個寬容度 (例如稍微放寬 20)
    # 但絕對不要超過 110 (經驗法則，超過就太亮了變成灰色/棕色)
    final_v_max = min(dynamic_v_max + 20, 110)

    # 策略 2: 飽和度 (S) 也要限制，避免抓到棕色
    # 黑點通常飽和度很低，我們取整體像素飽和度的中位數作為上限參考
    # 或者直接給一個經驗上的寬鬆上限 (例如 150)
    final_s_max = 150

    print(f"[Auto-Detect] 自動計算出的閾值 -> 亮度 V < {final_v_max:.2f}, 飽和度 S < {final_s_max}")

    return int(final_v_max), int(final_s_max)


In [20]:
def process_her2_dish_image_auto(img_path):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ... (前面的 Color Deconvolution 和 StarDist 保持不變) ...
    hed = rgb2hed(img_rgb)
    nuclei_channel = hed[:, :, 0]
    her2_membrane_channel = hed[:, :, 2]
    nuclei_img_vis = (rescale_intensity(nuclei_channel, out_range=(0, 255))).astype(np.uint8)
    her2_membrane_vis = (rescale_intensity(her2_membrane_channel, out_range=(0, 255))).astype(np.uint8)

    model = StarDist2D.from_pretrained('2D_versatile_he')
    img_input = np.stack([nuclei_img_vis]*3, axis=-1)
    labels, _ = model.predict_instances(normalize(img_input))

    _, her2_positive_mask = cv2.threshold(her2_membrane_vis, 100, 255, cv2.THRESH_BINARY)

    # ---------------------------------------------------------
    # 自動化訊號偵測 (Auto Thresholding)
    # ---------------------------------------------------------
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # 1. 自動計算黑點閾值
    auto_v_max, auto_s_max = get_dynamic_black_threshold(hsv, percentile=6) # 設 1.0% 通常能抓到所有黑點

    lower_black_auto = np.array([0, 0, 0])
    upper_black_auto = np.array([180, auto_s_max, auto_v_max]) # 使用自動算出的 V 和 S

    mask_black = cv2.inRange(hsv, lower_black_auto, upper_black_auto)

    # Debug 存圖確認
    cv2.imwrite("process/output/debug_mask_black_auto.jpg", mask_black)

    # 2. 紅點通常顏色比較固定，可以用寬鬆的固定範圍，或者也用類似邏輯抓 Hue
    # 這裡維持稍微放寬的固定範圍即可，因為紅色變異較小
    mask_red1 = cv2.inRange(hsv, (0, 40, 40), (15, 255, 255))
    mask_red2 = cv2.inRange(hsv, (160, 40, 40), (180, 255, 255))
    mask_red = cv2.bitwise_or(mask_red1, mask_red2)

    # ... (後面的計算迴圈保持不變，記得用 dilate) ...
    results = []
    for nuclei_id in np.unique(labels):
        if nuclei_id == 0: continue

        nucleus_mask = (labels == nuclei_id).astype(np.uint8) * 255
        kernel = np.ones((5,5), np.uint8)
        dilated_nucleus_mask = cv2.dilate(nucleus_mask, kernel, iterations=3)

        overlap = cv2.bitwise_and(her2_positive_mask, her2_positive_mask, mask=dilated_nucleus_mask)

        # 這裡可以設寬鬆一點，例如 > 0 就當作沾到陽性區域
        if cv2.countNonZero(overlap) > 10:
            roi_black = cv2.bitwise_and(mask_black, mask_black, mask=dilated_nucleus_mask)
            roi_red = cv2.bitwise_and(mask_red, mask_red, mask=dilated_nucleus_mask)

            n_her2 = max(0, cv2.connectedComponents(roi_black)[0] - 1)
            n_cep17 = max(0, cv2.connectedComponents(roi_red)[0] - 1)

            results.append({
                "id": nuclei_id,
                "her2_dots": n_her2,
                "cep17_dots": n_cep17,
                "ratio": n_her2 / n_cep17 if n_cep17 > 0 else 0
            })
    return results, mask_black

In [21]:
# 執行
res, mask = process_her2_dish_image_auto('process/tile_x153600_y59392.tiff')
print(res)

Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
[Auto-Detect] 自動計算出的閾值 -> 亮度 V < 110.00, 飽和度 S < 150
[{'id': 11, 'her2_dots': 2, 'cep17_dots': 1, 'ratio': 2.0}, {'id': 14, 'her2_dots': 4, 'cep17_dots': 1, 'ratio': 4.0}, {'id': 40, 'her2_dots': 2, 'cep17_dots': 32, 'ratio': 0.0625}, {'id': 48, 'her2_dots': 1, 'cep17_dots': 3, 'ratio': 0.3333333333333333}, {'id': 66, 'her2_dots': 1, 'cep17_dots': 2, 'ratio': 0.5}, {'id': 70, 'her2_dots': 1, 'cep17_dots': 17, 'ratio': 0.058823529411764705}, {'id': 73, 'her2_dots': 2, 'cep17_dots': 2, 'ratio': 1.0}, {'id': 89, 'her2_dots': 3, 'cep17_dots': 9, 'ratio': 0.3333333333333333}, {'id': 138, 'her2_dots': 1, 'cep17_dots': 19, 'ratio': 0.05263157894736842}, {'id': 143, 'her2_dots': 4, 'cep17_dots': 6, 'ratio': 0.6666666666666666}, {'id': 157, 'her2_dots': 1, 'cep17_dots': 9, 'ratio': 0.111111111111111

In [22]:
import os

def save_results_to_images(img_path, results, output_dir='process/output/id'):
    """
    將每顆細胞核的分析結果裁切成單獨圖片並儲存。

    Parameters:
    -----------
    img_path : str
        原始圖片路徑
    results : list of dict
        分析結果，包含 id, her2_dots, cep17_dots, ratio
    output_dir : str
        輸出目錄路徑
    """
    # 讀取原圖
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 重新計算 labels (取得細胞核分割結果)
    hed = rgb2hed(img_rgb)
    nuclei_channel = hed[:, :, 0]
    nuclei_img_vis = (rescale_intensity(nuclei_channel, out_range=(0, 255))).astype(np.uint8)

    model = StarDist2D.from_pretrained('2D_versatile_he')
    img_input = np.stack([nuclei_img_vis]*3, axis=-1)
    labels, _ = model.predict_instances(normalize(img_input))

    # 如果目錄存在，先刪除再建立
    if os.path.exists(output_dir):
        import shutil
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # 處理每個結果
    for cell_info in results:
        cell_id = cell_info['id']
        her2_dots = cell_info['her2_dots']
        cep17_dots = cell_info['cep17_dots']
        ratio = cell_info['ratio']

        # 取得該細胞核的遮罩
        nucleus_mask = (labels == cell_id).astype(np.uint8) * 255

        # 找出細胞核的 bounding box
        contours, _ = cv2.findContours(nucleus_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) == 0:
            continue

        x, y, w, h = cv2.boundingRect(contours[0])

        # 加一點 padding
        padding = 20
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(img.shape[1], x + w + padding)
        y2 = min(img.shape[0], y + h + padding)

        # 裁切圖片
        cropped_img = img[y1:y2, x1:x2].copy()

        # 在圖片上標註資訊
        text_lines = [
            f"ID: {cell_id}",
            f"HER2: {her2_dots}",
            f"CEP17: {cep17_dots}",
            f"Ratio: {ratio:.2f}"
        ]

        # 計算文字區域的背景
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.4
        thickness = 1
        line_height = 15

        # 在圖片下方加上文字資訊
        # 先擴展圖片高度以放文字
        text_area_height = len(text_lines) * line_height + 10
        new_height = cropped_img.shape[0] + text_area_height
        extended_img = np.ones((new_height, cropped_img.shape[1], 3), dtype=np.uint8) * 255
        extended_img[:cropped_img.shape[0], :] = cropped_img

        # 寫入文字
        for i, text in enumerate(text_lines):
            y_pos = cropped_img.shape[0] + 12 + i * line_height
            cv2.putText(extended_img, text, (5, y_pos), font, font_scale, (0, 0, 0), thickness)

        # 儲存圖片
        output_path = os.path.join(output_dir, f"cell_{cell_id}.png")

        # 先刪除舊檔案(若存在)
        if os.path.exists(output_path):
            os.remove(output_path)

        cv2.imwrite(output_path, extended_img)
        print(f"已儲存: {output_path}")

    print(f"\n總共儲存 {len(results)} 張圖片到 {output_dir}")
    return output_dir


In [23]:
# 測試：執行分析並儲存圖片
res, mask = process_her2_dish_image_auto('process/tile_x153600_y59392.tiff')
save_results_to_images('process/tile_x153600_y59392.tiff', res)

Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
[Auto-Detect] 自動計算出的閾值 -> 亮度 V < 110.00, 飽和度 S < 150
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
已儲存: process/output/id\cell_11.png
已儲存: process/output/id\cell_14.png
已儲存: process/output/id\cell_40.png
已儲存: process/output/id\cell_48.png
已儲存: process/output/id\cell_66.png
已儲存: process/output/id\cell_70.png
已儲存: process/output/id\cell_73.png
已儲存: process/output/id\cell_89.png
已儲存: process/output/id\cell_138.png
已儲存: process/output/id\cell_143.png
已儲存: process/output/id\cell_157.png
已儲存: process/output/id\cell_159.png
已儲存: process/output/id\cell_186.png
已儲存: process/output/id\cell_210.png

總共儲存 14 張圖片到 process/output/id


'process/output/id'